# Medicare data download
Run each cell in order by pressing the play button on each. No Google Drive needed: the last cell downloads a zip straight to your computer.

In [ ]:
# Step 1: install DuckDB
!pip -q install duckdb

In [ ]:
# Step 2: settings and helper functions
import shutil
from pathlib import Path

import duckdb
import requests

OUT = Path("/content/medicare_data")
RAW = OUT / "raw"
PQ = OUT / "interim"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"
    )
}

SERIES = {
    "partb_provider": "Medicare Physician & Other Practitioners - by Provider",
    "partd_provider": "Medicare Part D Prescribers - by Provider",
}

LEIE_URL = "https://oig.hhs.gov/exclusions/downloadables/UPDATED.csv"


def series_title(dataset):
    for s in dataset.get("inSeries") or []:
        if s.get("title"):
            return s["title"].strip()
    return dataset.get("title", "").split(" : ")[0].strip()


def csv_links(datasets, name):
    links = {}
    for d in datasets:
        if series_title(d) != name:
            continue
        end = (d.get("temporal") or [{}])[0].get("endDate", "")
        if not end[:4].isdigit():
            continue
        for dist in d.get("distribution", []):
            url = dist.get("downloadURL", "")
            if dist.get("format") == "CSV" and url.lower().endswith(".csv"):
                links[int(end[:4])] = url
    return links


def download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    print("downloading", dest.name)
    with requests.get(url, headers=HEADERS, stream=True, timeout=900) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)


print("Settings loaded")

In [ ]:
# Step 3: find the available years in the CMS catalog
response = requests.get("https://data.cms.gov/data.json", headers=HEADERS, timeout=300)
response.raise_for_status()
datasets = response.json()["dataset"]

all_links = {}
for key, name in SERIES.items():
    all_links[key] = csv_links(datasets, name)
    print(key, "years available:", sorted(all_links[key]))

common_years = set.intersection(*(set(v) for v in all_links.values()))
years = sorted(common_years)[-3:]
print("Will download years:", years)

In [ ]:
# Step 4: download the files (takes 10-20 minutes)
for key, links in all_links.items():
    for year in years:
        download(links[year], RAW / key / f"{key}_{year}.csv")

download(LEIE_URL, RAW / "leie" / "leie_updated.csv")
print("All downloads finished")

In [ ]:
# Step 5: convert to Parquet and count rows
con = duckdb.connect()

for csv_path in sorted(RAW.rglob("*.csv")):
    pq_path = PQ / csv_path.parent.name / (csv_path.stem + ".parquet")
    pq_path.parent.mkdir(parents=True, exist_ok=True)

    con.execute(f"""
        COPY (
            SELECT * FROM read_csv('{csv_path}', header = true,
                                   all_varchar = true, normalize_names = true)
        ) TO '{pq_path}' (FORMAT parquet, COMPRESSION zstd)
    """)

    rows = con.execute(f"SELECT count(*) FROM '{pq_path}'").fetchone()[0]
    print(f"{pq_path.relative_to(OUT)}: {rows:,} rows")

In [ ]:
# Step 6: zip the Parquet files and download them to your computer
import shutil
from google.colab import files

shutil.make_archive("/content/medicare_interim", "zip", OUT, "interim")
print("Zip created, download starting...")
files.download("/content/medicare_interim.zip")